In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("data/processed/sales_clean.csv", parse_dates=["order_date"])

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset shape: (957, 11)


,order_id,order_date,customer,region,product,category,quantity,unit_price,discount,status,revenue
0,1010,2026-05-01,CUST-0035,East,Filing Cabinet,Furniture,4,249.0,0.10,Cancelled,896.4
1,1286,2026-04-11,CUST-0046,West,Docking Station,Electronics,13,219.0,0.00,Cancelled,2847.0
2,1675,2026-02-13,CUST-0136,West,Office Chair,Furniture,20,1099.0,0.00,Pending,21980.0
3,1100,2026-03-06,CUST-0011,East,Office Chair,Furniture,2,1099.0,0.15,Shipped,1868.3
4,1668,2026-05-12,CUST-0039,East,Filing Cabinet,Furniture,6,249.0,0.05,Pending,1419.3


In [3]:
print("Column Names:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

Column Names:
['order_id', 'order_date', 'customer', 'region', 'product', 'category', 'quantity', 'unit_price', 'discount', 'status', 'revenue']

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 957 entries, 0 to 956
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    957 non-null    int64         
 1   order_date  957 non-null    datetime64[us]
 2   customer    957 non-null    str           
 3   region      957 non-null    str           
 4   product     957 non-null    str           
 5   category    957 non-null    str           
 6   quantity    957 non-null    int64         
 7   unit_price  957 non-null    float64       
 8   discount    957 non-null    float64       
 9   status      957 non-null    str           
 10  revenue     957 non-null    float64       
dtypes: datetime64[us](1), float64(3), int64(2), str(5)
memory usage: 82.4 KB


In [4]:
total_revenue = df["revenue"].sum()

print("Q1: Total Revenue")
print(total_revenue)

Q1: Total Revenue
4162140.15


In [5]:
sales = df[df["status"] != "Cancelled"]

print("Original rows:", len(df))
print("Rows excluding cancelled:", len(sales))

Original rows: 957
Rows excluding cancelled: 811


In [6]:
revenue_by_region = (
    sales.groupby("region")["revenue"].sum().sort_values(ascending=False)
)

print("Q2: Revenue by Region (Highest First)")
print(revenue_by_region)

Q2: Revenue by Region (Highest First)
region
North    1112741.79
South     903500.33
East      825570.04
West      653020.08
Name: revenue, dtype: float64


In [7]:
top_5_products = (
    sales.groupby("product")["revenue"].sum().sort_values(ascending=False).head(5)
)

print("Q3: Top 5 Products by Revenue")
print(top_5_products)

Q3: Top 5 Products by Revenue
product
Standing Desk               1043584.05
Noise-Cancelling Headset     861123.05
Office Chair                 826448.00
Monitor 27in                 212846.55
Filing Cabinet               189837.60
Name: revenue, dtype: float64


In [8]:
average_by_category = (
    sales.groupby("category")["revenue"].mean().sort_values(ascending=False)
)

print("Q4: Average Order Value by Category")
print(average_by_category)

Q4: Average Order Value by Category
category
Furniture      7144.576792
Electronics    4814.161070
Accessories     392.038826
Name: revenue, dtype: float64


In [9]:
monthly_revenue = sales.groupby(sales["order_date"].dt.to_period("M"))["revenue"].sum()

print("Q5: Monthly Revenue")
print(monthly_revenue)

Q5: Monthly Revenue
order_date
2026-01    562960.50
2026-02    629822.90
2026-03    584532.97
2026-04    516067.75
2026-05    619213.47
2026-06    582234.65
Freq: M, Name: revenue, dtype: float64


In [10]:
highest_revenue_month = monthly_revenue.idxmax()

print("Q6: Month with Highest Revenue")
print(highest_revenue_month)

Q6: Month with Highest Revenue
2026-02


In [11]:
cancelled_by_region = (
    df.assign(is_cancelled=df["status"] == "Cancelled")
    .groupby("region")["is_cancelled"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print("Q7: Cancelled Orders Percentage by Region")
print(cancelled_by_region)

Q7: Cancelled Orders Percentage by Region
region
East     18.032787
West     15.675676
South    14.977974
North    12.956811
Name: is_cancelled, dtype: float64


In [12]:
top_5_customers = df[df["customer"] != "Unknown"]["customer"].value_counts().head(5)

print("Q8: Top 5 Customers by Number of Orders")
print(top_5_customers)

Q8: Top 5 Customers by Number of Orders
customer
CUST-0124    14
CUST-0142    13
CUST-0069    12
CUST-0009    12
CUST-0023    11
Name: count, dtype: int64


In [13]:
average_discount = (
    df.groupby("category")["discount"].mean().sort_values(ascending=False)
)

print("Q9: Average Discount by Category")
print(average_discount)

Q9: Average Discount by Category
category
Accessories    0.041379
Electronics    0.041277
Furniture      0.037717
Name: discount, dtype: float64


In [14]:
revenue_pivot = sales.pivot_table(
    index="region", columns="category", values="revenue", aggfunc="sum"
)

print("Q10: Revenue by Region and Category")
print(revenue_pivot)

Q10: Revenue by Region and Category
category  Accessories  Electronics  Furniture
region                                       
East         20623.59    389392.85  415553.60
North        34315.34    421877.15  656549.30
South        23182.53    281144.55  599173.25
West         18712.13    212223.10  422084.85


In [15]:
products = pd.read_csv("data/raw/products.csv")

print("Product Data:")
display(products.head())

print("\nProduct Columns:")
print(products.columns.tolist())

Product Data:


,product,cost_price,supplier
0,USB-C Cable,7.38,Apex Supply
1,Wireless Mouse,16.32,BrightTech
2,Keyboard Pro,56.96,BrightTech
3,Laptop Stand,27.90,Nordic Goods
4,Webcam HD,79.98,Apex Supply



Product Columns:
['product', 'cost_price', 'supplier']


In [16]:
joined = sales.merge(products, on="product", how="left")

print("Original sales rows:", len(sales))
print("Joined sales rows:", len(joined))

display(joined.head())

Original sales rows: 811
Joined sales rows: 811


,order_id,order_date,customer,region,product,category,quantity,unit_price,discount,status,revenue,cost_price,supplier
0,1675,2026-02-13,CUST-0136,West,Office Chair,Furniture,20,1099.0,0.00,Pending,21980.00,747.32,BrightTech
1,1100,2026-03-06,CUST-0011,East,Office Chair,Furniture,2,1099.0,0.15,Shipped,1868.30,747.32,BrightTech
2,1668,2026-05-12,CUST-0039,East,Filing Cabinet,Furniture,6,249.0,0.05,Pending,1419.30,181.77,BrightTech
3,1644,2026-06-19,CUST-0039,North,Noise-Cancelling Headset,Electronics,20,1249.0,0.05,Shipped,23731.00,736.91,Nordic Goods
4,1172,2026-03-14,CUST-0132,North,Webcam HD,Electronics,17,129.0,0.15,Pending,1864.05,79.98,Apex Supply


In [17]:
joined["profit"] = joined["revenue"] - (joined["quantity"] * joined["cost_price"])

print("Q11: Total Profit")
print(round(joined["profit"].sum(), 2))

Q11: Total Profit
1160800.32


In [18]:
top_3_products_profit = (
    joined.groupby("product")["profit"].sum().sort_values(ascending=False).head(3)
)

print("Q12: Top 3 Products by Total Profit")
print(top_3_products_profit)

Q12: Top 3 Products by Total Profit
product
Standing Desk               379646.63
Noise-Cancelling Headset    326126.39
Office Chair                248022.32
Name: profit, dtype: float64


In [19]:
revenue_by_weekday = (
    sales.groupby(sales["order_date"].dt.day_name())["revenue"]
    .sum()
    .sort_values(ascending=False)
)

print("Q13: Revenue by Weekday")
print(revenue_by_weekday)

Q13: Revenue by Weekday
order_date
Friday       605133.66
Monday       578727.93
Saturday     534738.13
Tuesday      470372.49
Sunday       454729.90
Thursday     432210.51
Wednesday    418919.62
Name: revenue, dtype: float64


In [20]:
orders_by_status = df["status"].value_counts()

print("Q14: Number of Orders per Status")
print(orders_by_status)

Q14: Number of Orders per Status
status
Shipped      657
Pending      154
Cancelled    146
Name: count, dtype: int64


In [21]:
average_discount_by_region = (
    df.groupby("region")["discount"].mean().sort_values(ascending=False)
)

print("Q15: Average Discount by Region")
print(average_discount_by_region)

Q15: Average Discount by Region
region
East     0.040984
North    0.040698
South    0.039427
West     0.038378
Name: discount, dtype: float64


## Business Insights

1. **Q1:** Total revenue from all orders is 4,162,140.15.

2. **Q2:** North generated the highest revenue among all regions, excluding cancelled orders.

3. **Q3:** Standing Desk was the highest-revenue product, followed by Noise-Cancelling Headset and Office Chair.

4. **Q4:** Furniture had the highest average order value, excluding cancelled orders.

5. **Q5:** February 2026 generated the highest monthly revenue.

6. **Q6:** February 2026 was the best-performing month in terms of revenue.

7. **Q7:** East had the highest percentage of cancelled orders among the four regions.

8. **Q8:** CUST-0124 had the highest number of orders among known customers.

9. **Q9:** Accessories had the highest average discount among the three categories.

10. **Q10:** North generated the highest revenue overall, with Furniture contributing the most revenue across regions.

11. **Q11:** Total profit from non-cancelled orders was 1,160,800.32.

12. **Q12:** Standing Desk generated the highest total profit, followed by Noise-Cancelling Headset and Office Chair.

13. **Q13:** Friday generated the highest revenue, while Wednesday generated the lowest.

14. **Q14:** Out of 957 orders, 657 were Shipped, 154 were Pending, and 146 were Cancelled.

15. **Q15:** East had the highest average discount at approximately 4.10%, while West had the lowest at approximately 3.84%.